## 1. Dekoratory

**Zadanie 1**

In [1]:
import functools

In [2]:
def show_list_length(func):
    @functools.wraps(func)
    def opakowanie(*args, **kwargs):
        for rzecz in args:
            if isinstance(rzecz, list):
                print(f"Liczba elementów listy: {len(rzecz)}")

        for nazwa, rzecz in kwargs.items():
            if isinstance(rzecz, list):
                print(f"Liczba elementów listy '{nazwa}': {len(rzecz)}")

        return func(*args, **kwargs)

    return opakowanie

In [3]:
@show_list_length
def process_data(data_list, name):
    print(f"Przetwarzanie {name}")


process_data([1, 2, 3, 4], "dane testowe")

Liczba elementów listy: 4
Przetwarzanie dane testowe


**Zadanie 2**

In [4]:
from datetime import datetime
import time

In [5]:
def logger(filename):
    def dekoruj(funkcja):
        @functools.wraps(funkcja)
        def srodek(*args, **kwargs):
            start = time.time()

            wynik = funkcja(*args, **kwargs)

            koniec = time.time()
            czas = koniec - start

            data = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            with open(filename, "a", encoding="utf-8") as plik:
                plik.write(
                    f"{data} | funkcja: {funkcja.__name__} | czas wykonania: {czas:.4f} s\n"
                )

            return wynik

        return srodek

    return dekoruj

In [6]:
@logger("moj_log.log")
def policz_cos():
    time.sleep(1)
    print("Liczenie zakończone")


policz_cos()

Liczenie zakończone


## 2. Deskryptory

**Zadanie 3**

In [8]:
import logging

logging.basicConfig(
    filename="dostep.log",
    level=logging.INFO,
    format="%(asctime)s | %(message)s",
    encoding="utf-8"
)

In [9]:
class AccessLogger:
    def __init__(self, nazwa_atrybutu):
        self.nazwa_atrybutu = nazwa_atrybutu
        self.nazwa_prywatna = "_" + nazwa_atrybutu

    def __get__(self, obiekt, klasa):
        if obiekt is None:
            return self

        wartosc = getattr(obiekt, self.nazwa_prywatna)
        logging.info(f"GET: odczytano '{self.nazwa_atrybutu}' = {wartosc}")

        return wartosc

    def __set__(self, obiekt, wartosc):
        logging.info(f"SET: ustawiono '{self.nazwa_atrybutu}' = {wartosc}")
        setattr(obiekt, self.nazwa_prywatna, wartosc)

In [10]:
class Uzytkownik:
    imie = AccessLogger("imie")
    wiek = AccessLogger("wiek")

    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek

In [13]:
u1 = Uzytkownik("Kasia", 22)

print(u1.imie, u1.wiek)

u1.imie = "Ola"
u1.wiek = 23

print(u1.imie, u1.wiek)

Kasia 22
Ola 23


## 3. Generatory i Iteratory

**Zadanie 4**

In [14]:
def collatz_generator(n):
    if n < 1:
        raise ValueError("n musi być liczbą > 0")

    while n != 1:
        yield n

        if n % 2 == 0:
            n = n // 2
        else:
            n = 3 * n + 1

    yield 1

In [15]:
for status in collatz_generator(10):
    print(status)

10
5
16
8
4
2
1


## Zadania do zrobienia w domu

**Zadanie 1**

In [26]:
current_user = {"username": "admin", "role": "superuser"}

In [27]:
def require_role(role):
    def sprawdzacz(funkcja):
        @functools.wraps(funkcja)
        def wejscie(*args, **kwargs):
            if current_user.get("role") != role:
                raise PermissionError(
                    f"Brak dostępu. Wymagana rola: {role}, "
                    f"aktualna rola: {current_user.get('role')}"
                )

            return funkcja(*args, **kwargs)

        return wejscie

    return sprawdzacz

In [28]:
@require_role("superuser")
def panel_admina():
    print("Witamy w panelu admina")


panel_admina()

Witamy w panelu admina


In [29]:
@require_role("user")
def zwykla_strefa():
    print("Dostęp do zwykłej strefy")


zwykla_strefa()

PermissionError: Brak dostępu. Wymagana rola: user, aktualna rola: superuser

**Zadanie 2**

In [30]:
class Typed:
    def __init__(self, typ):
        self.typ = typ
        self.nazwa = None

    def __set_name__(self, klasa, nazwa):
        self.nazwa = "_" + nazwa

    def __get__(self, obiekt, klasa):
        if obiekt is None:
            return self

        return getattr(obiekt, self.nazwa)

    def __set__(self, obiekt, wartosc):
        if not isinstance(wartosc, self.typ):
            raise TypeError(
                f"Zły typ dla {self.nazwa[1:]}. "
                f"Oczekiwano: {self.typ.__name__}, "
                f"otrzymano: {type(wartosc).__name__}"
            )

        setattr(obiekt, self.nazwa, wartosc)

In [31]:
class Uzytkownik:
    imie = Typed(str)
    wiek = Typed(int)

    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek

In [32]:
u = Uzytkownik("Kasia", 22)

print(u.imie, u.wiek)

Kasia 22


In [33]:
u.wiek = "dwadzieścia dwa"
print(u.imie, u.wiek)

TypeError: Zły typ dla wiek. Oczekiwano: int, otrzymano: str

**Zadanie 3**

In [34]:
def prime_generator():
    liczba = 2

    while True:
        czy_pierwsza = True

        for dzielnik in range(2, int(liczba ** 0.5) + 1):
            if liczba % dzielnik == 0:
                czy_pierwsza = False
                break

        if czy_pierwsza:
            yield liczba

        liczba += 1

In [35]:
pierwsze_na_7 = (
    x for x in prime_generator()
    if x % 10 == 7
)

In [36]:
for _ in range(10):
    print(next(pierwsze_na_7))

7
17
37
47
67
97
107
127
137
157
